$$
\frac{\partial C}{\partial t}
+ u\frac{\partial C}{\partial x}
+ v\frac{\partial C}{\partial y}
= D\left(
\frac{\partial^2 C}{\partial x^2}
+ \frac{\partial^2 C}{\partial y^2}
\right)
$$

first i'll take a dye concentration field and a "pre solved velocity field" (Vortex flow in this case)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

In [ ]:
N = 300
L = 1.0

x = np.linspace(0, L, N, endpoint=False)
y = np.linspace(0, L, N, endpoint=False)

dx = L / N
dy = L / N

X,Y = np.meshgrid(x,y,indexing='ij')

In [ ]:
U0 = 5.0

u = U0 * np.sin(2 * np.pi * Y / L)
v = -U0 * np.sin(2 * np.pi * X / L)

In [ ]:
D = 1e-3
sigma = 0.06

C = np.exp(
    -((X - 0.25 * L)**2 + (Y - 0.50 * L)**2)
    / (2 * sigma**2)
)

Now we have to find the appropriate $\Delta t$ for our numerical scheme to ensure numerical stability. 

For a two dimensional explicit upwind scheme, a typical advection restriction is - 
$$\Delta t (\frac{|u|_{max}}{\Delta x} + \frac{|v|_{max}}{\Delta y})\leq 1$$

For the 2D diffusion equation, forward Euler time integration with central spatial differences requires

$$
\Delta t_{\mathrm{diff}}
\leq
\frac{1}{
2D\left(\frac{1}{\Delta x^2}+
\frac{1}{\Delta y^2}\right)}
$$

since both the restrictions must be satisfied simulatenously; we take their minimum

In [ ]:
dt_adv = 0.35 * min(dx, dy) / U0
dt_diff = 0.2 / (D * (1/dx**2 + 1/dy**2))

dt = min(dt_adv, dt_diff)

dt

In [ ]:
def advance(C):
    # Upwind first derivatives for advection
    dCdx = np.where(
        u >= 0,
        (C - np.roll(C, 1, axis=0)) / dx,
        (np.roll(C, -1, axis=0) - C) / dx
    )

    dCdy = np.where(
        v >= 0,
        (C - np.roll(C, 1, axis=1)) / dy,
        (np.roll(C, -1, axis=1) - C) / dy
    )

    # Central second derivatives for diffusion
    lapC = (
        (np.roll(C, -1, axis=0) - 2*C
         + np.roll(C, 1, axis=0)) / dx**2
        +
        (np.roll(C, -1, axis=1) - 2*C
         + np.roll(C, 1, axis=1)) / dy**2
    )

    # Advection-diffusion equation
    dCdt = -u*dCdx - v*dCdy + D*lapC

    return C + dt*dCdt


# np.roll is np.roll(array, shift, axis), so i basically shifts array along
# the specificed axis
# np.roll(C, 1, axis=1)	Gets the concentration from the previous column, \(C_{i,j-1}\) 


In [ ]:
# --------------------------------------------------
# 6. Animate dye transport
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    C.T,
    extent=[0, L, 0, L],
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=1,
    interpolation="bilinear"
)

fig.colorbar(im, ax=ax, label="Dye concentration C")
ax.set_xlabel("x")
ax.set_ylabel("y")

t = 0.0

def update(frame):
    global C, t

    # Several numerical time steps per animation frame
    for _ in range(4):
        C = advance(C)
        t += dt

    im.set_data(C.T)
    ax.set_title(f"Dye transport: t = {t:.3f}")
    return [im]

ani = FuncAnimation(
    fig, update, frames=250,
    interval=30, blit=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))


In [ ]:
cfl= lambda u, dt,dx : u*dt/dx 

np.max(cfl(u,dt,dx))


This specific rhombus shape is due to the numerical diffusion (artifical viscocity) that is present due to 1st order upwind scheme, 

In a 2D Cartesian grid where the flow has a velocity magnitude $V$ and moves at an angle $\theta$ relative to the x-axis, this numerical error creates "cross-wind" diffusion—artificial smearing that occurs strictly perpendicular to the actual flow direction. The coefficient for this false cross-wind diffusion is proportional to the product of the flow's x and y velocity components.Mathematically, this cross-wind diffusion error scales with $\sin(\theta)\cos(\theta)$. Using a standard trigonometric identity, this is equivalent to $\frac{1}{2}\sin(2\theta)$.

In [ ]:
# 2nd order upwind scheme
dt=dt/20
U0 = 5.0

u = U0 * np.sin(2 * np.pi * Y / L)
v = -U0 * np.sin(2 * np.pi * X / L)

D = 1e-3
sigma = 0.06

C = np.exp(
    -((X - 0.25 * L)**2 + (Y - 0.50 * L)**2)
    / (2 * sigma**2)
)

def compute_rhs(C):
    """Calculates the spatial derivatives to return dC/dt"""
    # 2nd-Order Upwind first derivatives for advection
    dCdx = np.where(
        u >= 0,
        (3*C - 4*np.roll(C, 1, axis=0) + np.roll(C, 2, axis=0)) / (2*dx),
        (-np.roll(C, -2, axis=0) + 4*np.roll(C, -1, axis=0) - 3*C) / (2*dx)
    )

    dCdy = np.where(
        v >= 0,
        (3*C - 4*np.roll(C, 1, axis=1) + np.roll(C, 2, axis=1)) / (2*dy),
        (-np.roll(C, -2, axis=1) + 4*np.roll(C, -1, axis=1) - 3*C) / (2*dy)
    )

    # Central second derivatives for diffusion
    lapC = (
        (np.roll(C, -1, axis=0) - 2*C + np.roll(C, 1, axis=0)) / dx**2
        +
        (np.roll(C, -1, axis=1) - 2*C + np.roll(C, 1, axis=1)) / dy**2
    )

    # Return the assembled Advection-Diffusion RHS
    return -u*dCdx - v*dCdy + D*lapC

def advance(C):
    """Advances one time step using 3rd-Order TVD Runge-Kutta (SSP-RK3)"""
    
    # RK3 Stage 1
    rhs1 = compute_rhs(C)
    C1 = C + dt * rhs1
    
    # RK3 Stage 2
    rhs2 = compute_rhs(C1)
    C2 = 0.75 * C + 0.25 * C1 + 0.25 * dt * rhs2
    
    # RK3 Stage 3
    rhs3 = compute_rhs(C2)
    C_new = (1.0/3.0) * C + (2.0/3.0) * C2 + (2.0/3.0) * dt * rhs3
    
    return C_new

In [ ]:
# --------------------------------------------------
# 1. Pre-compute the simulation frames
# --------------------------------------------------
frames_C = []
time_vals = []
t = 0.0

total_frames = 1000
steps_per_frame = 80

print("Simulating physics... please wait.")

# Save the initial state (Frame 0)
frames_C.append(C.copy())
time_vals.append(t)

# Run the math loop
for i in range(1, total_frames):
    for _ in range(steps_per_frame):
        C = advance(C)
        t += dt
        
    frames_C.append(C.copy()) # .copy() is mandatory here!
    time_vals.append(t)

print("Simulation complete! Rendering animation...")

# --------------------------------------------------
# 2. Animate the pre-computed frames
# --------------------------------------------------



fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    frames_C[0].T,
    extent=[0, L, 0, L],
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=1,
    interpolation="bilinear"
)

fig.colorbar(im, ax=ax, label="Dye concentration C")
ax.set_xlabel("x")
ax.set_ylabel("y")

# The update function now does ZERO math, it just grabs the saved array
def update(frame):
    im.set_data(frames_C[frame].T)
    ax.set_title(f"Dye transport: t = {time_vals[frame]:.3f}")
    return [im]

ani = FuncAnimation(
    fig, update, frames=total_frames,
    interval=30, blit=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))

This is a bit better but there's still some numerical viscocity in the $\theta =  {45}^*$ , so to counter this we should use TVD Flux Limiters 

In [ ]:
import numpy as np

# --------------------------------------------------
# TVD Helper Functions
# --------------------------------------------------
def van_leer(r):
    """van Leer flux limiter function."""
    # Returns 0 for negative r, smoothing sharp dispersive gradients
    return (r + np.abs(r)) / (1.0 + np.abs(r))

def tvd_derivative(C, vel, d, axis):
    """Calculates the 2nd-order TVD spatial derivative."""
    # Shift arrays to get neighboring cell values
    C_m2 = np.roll(C, 2, axis=axis)
    C_m1 = np.roll(C, 1, axis=axis)
    C_p1 = np.roll(C, -1, axis=axis)
    C_p2 = np.roll(C, -2, axis=axis)
    
    eps = 1e-12 # Prevent division by zero
    
    # --- Positive Velocity (Flow moving in the positive direction) ---
    del_i_pos   = C - C_m1       # Local gradient
    del_ip1_pos = C_p1 - C       # Downstream gradient
    del_im1_pos = C_m1 - C_m2    # Upstream gradient
    
    # Calculate ratio of consecutive gradients (r)
    r_i_pos   = np.where(np.abs(del_i_pos) < eps, 0.0, del_im1_pos / del_i_pos)
    r_ip1_pos = np.where(np.abs(del_ip1_pos) < eps, 0.0, del_i_pos / del_ip1_pos)
    
    # Apply flux limiter
    phi_i_pos   = van_leer(r_i_pos)
    phi_ip1_pos = van_leer(r_ip1_pos)
    
    # Assemble 1st-order upwind + limited 2nd-order correction
    dC_pos = del_i_pos/d + (0.5/d) * (phi_ip1_pos * del_ip1_pos - phi_i_pos * del_i_pos)
    
    # --- Negative Velocity (Flow moving in the negative direction) ---
    del_i_neg   = C_p1 - C       # Local gradient
    del_im1_neg = C - C_m1       # Downstream gradient
    del_ip1_neg = C_p2 - C_p1    # Upstream gradient
    
    # Calculate ratio of consecutive gradients (r) looking backwards
    r_i_neg   = np.where(np.abs(del_i_neg) < eps, 0.0, del_ip1_neg / del_i_neg)
    r_im1_neg = np.where(np.abs(del_im1_neg) < eps, 0.0, del_i_neg / del_im1_neg)
    
    # Apply flux limiter
    phi_i_neg   = van_leer(r_i_neg)
    phi_im1_neg = van_leer(r_im1_neg)
    
    # Assemble 1st-order upwind + limited 2nd-order correction
    dC_neg = del_i_neg/d + (0.5/d) * (phi_im1_neg * del_im1_neg - phi_i_neg * del_i_neg)
    
    # Select the correct derivative based on local velocity direction
    return np.where(vel >= 0, dC_pos, dC_neg)


# --------------------------------------------------
# Main Solver Math
# --------------------------------------------------
dt=dt*20
U0 = 5.0

u = U0 * np.sin(2 * np.pi * Y / L)
v = -U0 * np.sin(2 * np.pi * X / L)

D = 1e-3
sigma = 0.06

C = np.exp(
    -((X - 0.25 * L)**2 + (Y - 0.50 * L)**2)
    / (2 * sigma**2)
)

def compute_rhs(C):
    """Calculates the spatial derivatives to return dC/dt"""
    # Advection using TVD scheme
    dCdx = tvd_derivative(C, u, dx, axis=0)
    dCdy = tvd_derivative(C, v, dy, axis=1)

    # Central second derivatives for diffusion (remains unchanged)
    lapC = (
        (np.roll(C, -1, axis=0) - 2*C + np.roll(C, 1, axis=0)) / dx**2
        +
        (np.roll(C, -1, axis=1) - 2*C + np.roll(C, 1, axis=1)) / dy**2
    )

    # Return the assembled Advection-Diffusion RHS
    return -u*dCdx - v*dCdy + D*lapC

def advance(C):
    """Advances one time step using 3rd-Order TVD Runge-Kutta (SSP-RK3)"""
    # RK3 Stage 1
    rhs1 = compute_rhs(C)
    C1 = C + dt * rhs1
    
    # RK3 Stage 2
    rhs2 = compute_rhs(C1)
    C2 = 0.75 * C + 0.25 * C1 + 0.25 * dt * rhs2
    
    # RK3 Stage 3
    rhs3 = compute_rhs(C2)
    C_new = (1.0/3.0) * C + (2.0/3.0) * C2 + (2.0/3.0) * dt * rhs3
    
    return C_new

In [ ]:
# --------------------------------------------------
# 1. Pre-compute the simulation frames
# --------------------------------------------------
frames_C = []
time_vals = []
t = 0.0

total_frames = 1000
steps_per_frame = 4

print("Simulating physics... please wait.")

# Save the initial state (Frame 0)
frames_C.append(C.copy())
time_vals.append(t)

# Run the math loop
for i in range(1, total_frames):
    for _ in range(steps_per_frame):
        C = advance(C)
        t += dt
        
    frames_C.append(C.copy()) # .copy() is mandatory here!
    time_vals.append(t)

print("Simulation complete! Rendering animation...")

# --------------------------------------------------
# 2. Animate the pre-computed frames
# --------------------------------------------------

plt.rcParams['animation.embed_limit'] = 20


fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    frames_C[0].T,
    extent=[0, L, 0, L],
    origin="lower",
    cmap="turbo",
    vmin=0,
    vmax=1,
    interpolation="bilinear"
)

fig.colorbar(im, ax=ax, label="Dye concentration C")
ax.set_xlabel("x")
ax.set_ylabel("y")

# The update function now does ZERO math, it just grabs the saved array
def update(frame):
    im.set_data(frames_C[frame].T)
    ax.set_title(f"Dye transport: t = {time_vals[frame]:.3f}")
    return [im]

ani = FuncAnimation(
    fig, update, frames=total_frames,
    interval=30, blit=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))